# Sistema Especialista: Triagem Inteligente de Chamados de Infraestrutura de TI

**Disciplina:** Inteligência Artificial — Profa. Kadidja Valéria  
**Referencial Conceitual:** Guias de Sistemas Especialistas, Engenharia do Conhecimento e *Trustworthy AI* (IBM) + Boas Práticas de SLA (ITIL).

---

## Objetivo do Notebook
Implementar e validar um **Sistema Especialista baseado em Regras** capaz de realizar a triagem inicial de incidentes de infraestrutura de TI (Storage, RAID, Redes e Suporte N1), demonstrando:
1. **Separação arquitetural** entre **Base de Conhecimento** e **Motor de Inferência**;
2. **Encadeamento para Frente (*Forward Chaining*)** e **Encadeamento para Trás (*Backward Chaining*)**;
3. **Módulo de Explicação** transparente (caixa-branca);
4. **Execução e análise de 3 casos de teste** (incluindo tratamento de incerteza);
5. **Conexão com Aprendizado Supervisionado** (extração de regras via Árvore de Decisão).

In [1]:
# ==============================================================================
# 1. BASE DE CONHECIMENTO (Fatos/Sintomas, Regras SE-ENTÃO e Diagnósticos ITIL)
# ==============================================================================

PERGUNTAS = {
    "disco_falhou": "Algum disco físico do storage apresentou falha (LED laranja/alerta)?",
    "raid_degradado": "O array RAID está em modo degradado?",
    "volume_offline": "O volume/unidade de rede está offline (indisponível para os usuários)?",
    "alerta_smart": "O disco apresenta apenas alerta preventivo (SMART) sem falha física?",
    "sem_acesso_rede": "Há perda de acesso à rede local ou internet?",
    "varios_usuarios": "O problema afeta vários usuários ou um setor inteiro?",
    "apenas_um_usuario": "O problema acontece apenas no computador de um único usuário?",
}

REGRAS = [
    {
        "id": "R1",
        "se": ["disco_falhou", "raid_degradado"],
        "entao": "raid_em_risco",
        "descricao": "SE disco_falhou E raid_degradado ENTÃO raid_em_risco (fato intermediário)",
    },
    {
        "id": "R2",
        "se": ["raid_em_risco", "volume_offline"],
        "entao": "triagem_storage_critica",
        "descricao": "SE raid_em_risco E volume_offline ENTÃO Chamado Crítico de Storage",
    },
    {
        "id": "R3",
        "se": ["alerta_smart"],
        "entao": "triagem_storage_media",
        "descricao": "SE alerta_smart ENTÃO Chamado Médio de Storage (Troca Preventiva)",
    },
    {
        "id": "R4",
        "se": ["sem_acesso_rede", "varios_usuarios"],
        "entao": "triagem_rede_alta",
        "descricao": "SE sem_acesso_rede E varios_usuarios ENTÃO Chamado Alto de Rede Local",
    },
    {
        "id": "R5",
        "se": ["sem_acesso_rede", "apenas_um_usuario"],
        "entao": "triagem_suporte_media",
        "descricao": "SE sem_acesso_rede E apenas_um_usuario ENTÃO Chamado Médio de Suporte N1",
    },
]

DIAGNOSTICOS = {
    "triagem_storage_critica": {
        "categoria": "Storage / Armazenamento",
        "prioridade": "CRÍTICA",
        "sla": "Primeira resposta em 15 min | Solução em até 4 horas",
        "equipe": "Especialistas de Storage / Infraestrutura N3",
        "acao_inicial": "Bloquear gravações não essenciais, validar backup recente e preparar substituição imediata do disco.",
    },
    "triagem_storage_media": {
        "categoria": "Storage / Armazenamento",
        "prioridade": "MÉDIA",
        "sla": "Primeira resposta em 2 horas | Solução em até 24 horas",
        "equipe": "Infraestrutura N2",
        "acao_inicial": "Agendar janela de manutenção para troca preventiva do disco com alerta SMART.",
    },
    "triagem_rede_alta": {
        "categoria": "Redes",
        "prioridade": "ALTA",
        "sla": "Primeira resposta em 30 min | Solução em até 8 horas",
        "equipe": "Equipe de Redes",
        "acao_inicial": "Verificar status do switch/gateway do setor afetado antes de deslocar técnico.",
    },
    "triagem_suporte_media": {
        "categoria": "Estação de Trabalho / Acesso",
        "prioridade": "MÉDIA",
        "sla": "Primeira resposta em 2 horas | Solução em até 24 horas",
        "equipe": "Suporte Técnico N1",
        "acao_inicial": "Validar cabo de rede, configuração de IP/DNS e testar acesso remoto na máquina do usuário.",
    },
}

print(f"Base de Conhecimento carregada: {len(PERGUNTAS)} sintomas, {len(REGRAS)} regras e {len(DIAGNOSTICOS)} perfis de triagem.")

Base de Conhecimento carregada: 7 sintomas, 5 regras e 4 perfis de triagem.


In [2]:
# ==============================================================================
# 2. MOTOR DE INFERÊNCIA E MÓDULO DE EXPLICAÇÃO
# ==============================================================================

def encadeamento_para_frente(fatos_iniciais):
    '''
    Raciocínio guiado pelos dados (Data-Driven):
    Parte dos sintomas observados e dispara regras em ciclos até esgotar deduções.
    '''
    fatos = set(fatos_iniciais)
    regras_disparadas = []
    novo_fato = True

    while novo_fato:
        novo_fato = False
        for regra in REGRAS:
            if regra["entao"] not in fatos:
                if all(condicao in fatos for condicao in regra["se"]):
                    fatos.add(regra["entao"])
                    regras_disparadas.append(regra)
                    novo_fato = True

    diagnosticos = [f for f in fatos if f in DIAGNOSTICOS]
    return fatos, regras_disparadas, diagnosticos


def encadeamento_para_tras(objetivo, respostas_simuladas=None, fatos_conhecidos=None, fatos_negados=None, trilha=None, perguntas_feitas=None):
    '''
    Raciocínio guiado por objetivos (Goal-Driven):
    Parte de uma hipótese e consulta apenas os sintomas estritamente necessários.
    '''
    if fatos_conhecidos is None:
        fatos_conhecidos = set()
    if fatos_negados is None:
        fatos_negados = set()
    if trilha is None:
        trilha = []
    if perguntas_feitas is None:
        perguntas_feitas = []

    if objetivo in fatos_conhecidos:
        return True, trilha, perguntas_feitas
    if objetivo in fatos_negados:
        return False, trilha, perguntas_feitas

    regras_candidatas = [r for r in REGRAS if r["entao"] == objetivo]

    if regras_candidatas:
        for regra in regras_candidatas:
            todas_verdadeiras = True
            for condicao in regra["se"]:
                ok, _, _ = encadeamento_para_tras(
                    condicao, respostas_simuladas, fatos_conhecidos, fatos_negados, trilha, perguntas_feitas
                )
                if not ok:
                    todas_verdadeiras = False
                    break
            if todas_verdadeiras:
                fatos_conhecidos.add(objetivo)
                trilha.append(regra)
                return True, trilha, perguntas_feitas
        return False, trilha, perguntas_feitas

    if objetivo in PERGUNTAS:
        perguntas_feitas.append(objetivo)
        if respostas_simuladas is not None:
            resp = "s" if respostas_simuladas.get(objetivo, False) else "n"
        else:
            resp = input(f"  -> {PERGUNTAS[objetivo]} (s/n): ").strip().lower()

        if resp == "s":
            fatos_conhecidos.add(objetivo)
            return True, trilha, perguntas_feitas
        else:
            fatos_negados.add(objetivo)
            return False, trilha, perguntas_feitas

    return False, trilha, perguntas_feitas


def executar_analise_caso(titulo, sintomas_ativos, hipotese_teste):
    print("=" * 72)
    print(f"TESTE: {titulo}")
    print("=" * 72)
    print(f"Sintomas observados: {sintomas_ativos if sintomas_ativos else 'Nenhum sintoma crítico mapeado'}")

    # 1. Encadeamento para Frente
    fatos_finais, trilha_frente, diagnosticos = encadeamento_para_frente(sintomas_ativos)
    print("\n[1] ENCADEAMENTO PARA FRENTE (Forward Chaining):")
    if diagnosticos:
        for d in diagnosticos:
            info = DIAGNOSTICOS[d]
            print(f"  -> Categoria : {info['categoria']}")
            print(f"  -> Prioridade: {info['prioridade']}")
            print(f"  -> SLA       : {info['sla']}")
            print(f"  -> Equipe    : {info['equipe']}")
            print(f"  -> 1ª Ação   : {info['acao_inicial']}")
    else:
        print("  -> Resultado : INCONCLUSIVO (sintomas insuficientes na base de regras).")
        print("  -> Ação      : Encaminhar para análise manual de analista N2 (evita falso diagnóstico).")

    print("\n  [Trilha de Explicação]:")
    if trilha_frente:
        for idx, r in enumerate(trilha_frente, 1):
            print(f"     Passo {idx} ({r['id']}): {r['descricao']}")
    else:
        print("     Nenhuma regra disparada.")

    # 2. Encadeamento para Trás
    respostas_dict = {s: True for s in sintomas_ativos}
    confirmado, trilha_tras, perguntas_consultadas = encadeamento_para_tras(hipotese_teste, respostas_simuladas=respostas_dict)
    print(f"\n[2] ENCADEAMENTO PARA TRÁS (Testando hipótese: '{hipotese_teste}'):")
    print(f"  -> Hipótese confirmada? {'SIM' if confirmado else 'NÃO'}")
    print(f"  -> Perguntas necessárias: {len(perguntas_consultadas)} de {len(PERGUNTAS)} ({perguntas_consultadas})")
    print()

print("Motor de Inferência e Módulo de Explicação prontos.")

Motor de Inferência e Módulo de Explicação prontos.


## 3. Execução e Análise dos 3 Casos de Teste

Abaixo testamos três cenários representativos:
1. **Caso 1 (Storage Crítico):** Disparo encadeado em dois níveis (`R1` gera o fato intermediário `raid_em_risco` $\rightarrow$ `R2` conclui `triagem_storage_critica`).
2. **Caso 2 (Rede Setorial):** Diferenciação de impacto coletivo (`R4`, prioridade Alta) em contraste com estação individual (`R5`), além da economia de perguntas no encadeamento para trás (apenas 2 de 7 perguntas).
3. **Caso 3 (Sintomas Ambíguos / Incerteza):** Nenhum sintoma crítico mapeado na base; o motor recusa-se a "adivinhar" e recomenda escalonamento humano.

In [3]:
# ==============================================================================
# 3. EXECUÇÃO DOS 3 CASOS DE TESTE
# ==============================================================================

# CASO 1: Falha crítica em storage (disco falhou + RAID degradado + volume offline)
executar_analise_caso(
    titulo="CASO 1 — Falha Crítica de Storage Corporativo",
    sintomas_ativos=["disco_falhou", "raid_degradado", "volume_offline"],
    hipotese_teste="triagem_storage_critica"
)

# CASO 2: Queda de rede afetando múltiplos usuários de um departamento
executar_analise_caso(
    titulo="CASO 2 — Indisponibilidade de Rede Setorial",
    sintomas_ativos=["sem_acesso_rede", "varios_usuarios"],
    hipotese_teste="triagem_rede_alta"
)

# CASO 3: Sintoma vago / fora da base de conhecimento (lentidão intermitente sem falhas críticas)
executar_analise_caso(
    titulo="CASO 3 — Sintomas Ambíguos / Fora da Base (Tratamento de Incerteza)",
    sintomas_ativos=[],
    hipotese_teste="triagem_storage_critica"
)

TESTE: CASO 1 — Falha Crítica de Storage Corporativo
Sintomas observados: ['disco_falhou', 'raid_degradado', 'volume_offline']

[1] ENCADEAMENTO PARA FRENTE (Forward Chaining):
  -> Categoria : Storage / Armazenamento
  -> Prioridade: CRÍTICA
  -> SLA       : Primeira resposta em 15 min | Solução em até 4 horas
  -> Equipe    : Especialistas de Storage / Infraestrutura N3
  -> 1ª Ação   : Bloquear gravações não essenciais, validar backup recente e preparar substituição imediata do disco.

  [Trilha de Explicação]:
     Passo 1 (R1): SE disco_falhou E raid_degradado ENTÃO raid_em_risco (fato intermediário)
     Passo 2 (R2): SE raid_em_risco E volume_offline ENTÃO Chamado Crítico de Storage

[2] ENCADEAMENTO PARA TRÁS (Testando hipótese: 'triagem_storage_critica'):
  -> Hipótese confirmada? SIM
  -> Perguntas necessárias: 3 de 7 (['disco_falhou', 'raid_degradado', 'volume_offline'])

TESTE: CASO 2 — Indisponibilidade de Rede Setorial
Sintomas observados: ['sem_acesso_rede', 'varios_usua

## 4. Apoio de Aprendizado Supervisionado (Árvore de Decisão — Aula 13)

Para mitigar o **gargalo da aquisição de conhecimento**, demonstramos como uma `DecisionTreeClassifier` extrai automaticamente regras `SE... ENTÃO...` a partir de um histórico rotulado de chamados anteriores.

In [4]:
# ==============================================================================
# 4. CONEXÃO COM APRENDIZADO SUPERVISIONADO (ÁRVORE DE DECISÃO — AULA 13)
# ==============================================================================
from sklearn.tree import DecisionTreeClassifier, export_text

# Atributos: [disco_falhou, raid_degradado, volume_offline, varios_usuarios]
X_historico = [
    [1, 1, 1, 1],  # Crítica (0)
    [1, 1, 1, 0],  # Crítica (0)
    [1, 1, 0, 0],  # Alta (1)
    [0, 0, 0, 1],  # Alta (1)
    [0, 0, 0, 0],  # Média (2)
    [1, 0, 0, 0],  # Média (2)
]
y_prioridades = [0, 0, 1, 1, 2, 2]

nomes_atributos = ["disco_falhou", "raid_degradado", "volume_offline", "varios_usuarios"]
nomes_classes = ["Prioridade CRÍTICA", "Prioridade ALTA", "Prioridade MÉDIA"]

arvore = DecisionTreeClassifier(random_state=42)
arvore.fit(X_historico, y_prioridades)

print("=== REGRAS EXTRAÍDAS DO HISTÓRICO DE CHAMADOS (ÁRVORE DE DECISÃO) ===\n")
print(export_text(arvore, feature_names=nomes_atributos, class_names=nomes_classes))

=== REGRAS EXTRAÍDAS DO HISTÓRICO DE CHAMADOS (ÁRVORE DE DECISÃO) ===

|--- volume_offline <= 0.50
|   |--- varios_usuarios <= 0.50
|   |   |--- raid_degradado <= 0.50
|   |   |   |--- class: Prioridade MÉDIA
|   |   |--- raid_degradado >  0.50
|   |   |   |--- class: Prioridade ALTA
|   |--- varios_usuarios >  0.50
|   |   |--- class: Prioridade ALTA
|--- volume_offline >  0.50
|   |--- class: Prioridade CRÍTICA

